In [2]:
import os
import json
from tqdm import tqdm
import cobra
import modelseedpy
import cobrakbase
import pandas as pd
from cobrakbase.kbaseapi_cache import KBaseCache
config = cobra.Configuration()
# config.solver = 'cplex'

In [3]:
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')

In [4]:
genomes = {}
genome_assembly = {}
for info in tqdm(kbase.list_workspace(155805)):
    if info.type == 'KBaseGenomes.Genome':
        genome = kbase.get_from_ws(str(info))
        assembly = kbase.get_from_ws(genome.assembly_ref)
        genomes[info.id] = genome
        genome_assembly[info.id] = assembly
        #print(info.id, info.type)

100%|██████████| 1298/1298 [02:23<00:00,  9.03it/s]


In [5]:
_taxonomy = {}
_taxonomy.update(pd.read_csv('gtdb_table_arc.csv', index_col=0).to_dict()['Classification'])
_taxonomy.update(pd.read_csv('gtdb_table_bac.csv', index_col=0).to_dict()['Classification'])
genome_taxa = {}
for genome_id in genomes:
    genome_taxa[genome_id] = _taxonomy[(genome_assembly[genome_id].info.id)]

In [6]:
for genome_id in genome_taxa:
    taxa = genome_taxa[genome_id]
    _taxa_parts = taxa.split(';')
    if _taxa_parts[0] == 'd__Archaea':
        if 'f__Haloarculaceae' in taxa:
            print(taxa)

d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__CBA1134;s__CBA1134 sp009184545
d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Halomicroarcula;s__
d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Natronomonas;s__
d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Halomicroarcula;s__
d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Halomicroarcula;s__
d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Natronomonas;s__


In [63]:
models = {}
for genome_id in genome_taxa:
    taxa = genome_taxa[genome_id]
    _taxa_parts = taxa.split(';')
    if 'c__Halanaerobiia' in taxa:
        models[genome_id] = cobra.io.read_sbml_model(f'./data/models/base/{genome_id}.xml')
        print(taxa)

ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.


d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__CBA1134;s__CBA1134 sp009184545


ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.


d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Halomicroarcula;s__


ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.


...

d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Halomicroarcula;s__


ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.


d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Halomicroarcula;s__


ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.


d__Archaea;p__Halobacteriota;c__Halobacteria;o__Halobacteriales;f__Haloarculaceae;g__Natronomonas;s__


In [93]:
model_genomes = {}
for genome_id in genome_taxa:
    taxa = genome_taxa[genome_id]
    _taxa_parts = taxa.split(';')
    if 'f__Rhodobacteraceae' in taxa:
        model_genomes[genome_id] = genomes[genome_id]
        #models[genome_id] = cobra.io.read_sbml_model(f'./data/models/base/{genome_id}.xml')
        #print(taxa)

In [94]:
import pymongo
from modelseedpy_ext.re.hash_seq import HashSeq
mongo = pymongo.MongoClient('mongodb://sequoia.mcs.anl.gov:27017')
c_protein_to_rast = mongo['database']['protein_to_rast']
missing = {}
feature_genome_count = {}
genome_arr = list(model_genomes)
for genome_id in tqdm(genome_arr):
    genome = genomes[genome_id]
    for feature in genome.features:
        h = HashSeq(feature.seq)
        doc = c_protein_to_rast.find_one({'_id': h.hash_value})
        if doc:
            annotations_rast = doc['value']
            if annotations_rast:
                for annotation_rast in annotations_rast:
                    if annotation_rast not in feature_genome_count:
                        feature_genome_count[annotation_rast] = {genome_id: 0 for genome_id in genome_arr}
                    feature_genome_count[annotation_rast][genome_id] += 1
        else:
            missing[(genome_id, feature.id)] = feature
            #print(f'missing annotation {feature.id}')

100%|██████████| 25/25 [00:56<00:00,  2.26s/it]


In [95]:
pd.DataFrame(feature_genome_count).transpose().to_csv('./data/f__Rhodobacteraceae.rast.tsv', sep='\t')

In [84]:
all_feature_genome_count = feature_genome_count

In [103]:
for f in all_feature_genome_count:
    if 'coenzyme m' in f.lower():
        print(f)
        for g, v in all_feature_genome_count[f].items():
            if v != 0:
                print('\t', v, '\t', genome_taxa[g])

Methyl coenzyme M reductase system component A2
	 1 	 d__Archaea;p__Halobacteriota;c__Methanosarcinia;o__Methanosarcinales;f__Methanosarcinaceae;g__Methanolobus;s__
[Methyl-Co(III) methylamine-specific corrinoid protein]:coenzyme M methyltransferase (EC 2.1.1.247)
	 2 	 d__Archaea;p__Halobacteriota;c__Methanosarcinia;o__Methanosarcinales;f__Methanosarcinaceae;g__Methanolobus;s__
Methyl coenzyme M reductase system component A2 homolog
	 1 	 d__Archaea;p__Halobacteriota;c__Methanosarcinia;o__Methanosarcinales;f__Methanosarcinaceae;g__Methanolobus;s__
Methylcobalamin:coenzyme M methyltransferase MA0779
	 1 	 d__Archaea;p__Halobacteriota;c__Methanosarcinia;o__Methanosarcinales;f__Methanosarcinaceae;g__Methanolobus;s__
Methyl coenzyme M reductase gamma subunit (EC 2.8.4.1)
	 1 	 d__Archaea;p__Halobacteriota;c__Methanosarcinia;o__Methanosarcinales;f__Methanosarcinaceae;g__Methanolobus;s__
Methyl coenzyme M reductase alpha subunit (EC 2.8.4.1)
	 1 	 d__Archaea;p__Halobacteriota;c__Methanosarc

In [50]:
def crop_ast(seq):
    if seq[-1] == '*':
        return seq[:-1]
    return seq
h_seq = {}
for feature in missing.values():
    seq = crop_ast(feature.seq)
    h = HashSeq(seq).hash_value
    if h not in h_seq:
        h_seq[h] = seq 

In [52]:
def chunk_dict(data, c_protein_to_rast, chunk_size):
    """Yield successive chunks of size chunk_size from a dictionary."""
    payload = []
    for k, v in data.items():
        doc = c_protein_to_rast.find_one({'_id': k})
        if doc is None:
            payload.append(MSFeature(k, v))
            if len(payload) >= chunk_size:
                yield payload
                payload = []
        else:
            #print(doc)
            pass
    if len(payload) > 0:
        yield payload

In [54]:
from modelseedpy.core.msgenome import MSGenome, MSFeature
from modelseedpy import RastClient
rast = RastClient()
for chunk in tqdm(chunk_dict(h_seq, c_protein_to_rast, 5000)):
    genome = MSGenome()
    genome.features += chunk
    rast.annotate_genome(genome)
    for f in genome.features:
        c_protein_to_rast.insert_one({'_id': f.id, 'value': f.ontology_terms.get('RAST')})

31it [03:25,  6.64s/it]


In [51]:
model_id = 'Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST'


'Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST'

In [54]:
modelseed = modelseedpy.biochem.from_local('/scratch/fliu/data/ModelSEEDDatabase/')

In [62]:
template_arc = kbase.get_from_ws('ArchaeaModelTemplateV5_test', 12218)

In [89]:
super_model = cobra.core.Model()
super_model_reactions = {}
for model_id in models:
    model = models[model_id]
    for r in model.reactions:
        seed_id = r.id[:-1]
        if seed_id in template_arc.reactions:
            template_reaction = template_arc.reactions.get_by_id(seed_id)
            if template_reaction.id not in super_model_reactions:
                model_reaction = template_reaction.to_reaction(model, index='0')
                super_model_reactions[model_reaction.id] = model_reaction

In [90]:
super_model.add_reactions([r for r in super_model_reactions.values()])

In [91]:
clade = 'f__Haloferacaceae'
cobra.io.save_json_model(super_model, f'./data/models/{clade}.model.base.json')

In [99]:
for model_id in models:
    model = models[model_id]
    if 'cpd01024_c0' in model.metabolites:
        print(model_id)

Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_metabat.16.contigs__.RAST


In [100]:
models

{'Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.29.contigs__.RAST': <Model  at 0x7f238836e4a0>,
 'Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_metabat.14.contigs__.RAST': <Model  at 0x7f234618f670>,
 'Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.19.contigs__.RAST': <Model  at 0x7f2344bc69b0>,
 'Salt_Pond_MetaG_R1_C_H2O_MG_DASTool_bins_metabat.29.contigs__.RAST': <Model  at 0x7f234450d780>,
 'Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.17.contigs__.RAST': <Model  at 0x7f23440879a0>,
 'Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST': <Model  at 0x7f2343af6800>,
 'Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.8.contigs__.RAST': <Model  at 0x7f234329b130>,
 'Salt_Pond_MetaG_R2_B_H2O_MG_DASTool_bins_metabat.16.contigs__.RAST': <Model  at 0x7f2342bc3a60>,
 'Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.9.contigs__.RAST': <Model  at 0x7f2342569270>,
 'Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.35.contigs__.RAST': <Model  at 0x7f2341edece0>,

In [57]:
modelseed.reactions.get_by_id()

Reaction identifier,rxn00001
Name,diphosphate phosphohydrolase
Memory address,0x7f232ada7cd0
Stoichiometry,cpd00001_0 + cpd00012_0 --> 2 cpd00009_0 + cpd00067_0 H2O + PPi --> 2 Phosphate + H+
GPR,
Lower bound,0
Upper bound,1000


# OLD!

In [4]:
model = kbase.get_from_ws('Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.16.pangenome.RAST.glm4ec.pyr.O2.gf', 186678)

In [39]:
(model.reactions.EX_cpd25960_c0 *-1) + model.reactions.rxn26456_c0 + model.reactions.rxn26457_c0 + model.reactions.rxn43120_c0

Reaction identifier,EX_cpd25960_c0
Name,exchange for MePn
Memory address,0x7f2ba6fd9e10
Stoichiometry,cpd00001_c0 + cpd00002_c0 + cpd00005_c0 + cpd00017_c0 <=> cpd00006_c0 + cpd00012_c0 + cpd00060_c0 + 2 cpd00067_c0 + cpd00128_c0 + cpd01024_c0 + cpd03091_c0 + cpd26508_c0 H2O [c0] + ATP [c0] + NADPH [c0] + S-Adenosyl-L-methionine [c0] <=> NADP [c0] + PPi [c0] + L-Methionine [c0] + 2 H+ [c0] + Adenine [c0] + Methane [c0] + 5'-Deoxyadenosine [c0] + 5-phospho-alpha-D-...
GPR,(((Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.16.contigs__.RAST.CDS.2377 or...
Lower bound,-1000
Upper bound,1000


In [62]:
model.add_reactions([template.reactions.rxn26447_c.to_reaction(model)])

In [58]:
model.reactions.rxn43120_c0

Reaction identifier,rxn43120_c0
Name,rxn43120_c [c0]
Memory address,0x7f2ba8f1d360
Stoichiometry,"cpd00005_c0 + cpd00017_c0 + cpd26519_c0 --> cpd00006_c0 + cpd00060_c0 + cpd01024_c0 + cpd03091_c0 + cpd26508_c0 NADPH [c0] + S-Adenosyl-L-methionine [c0] + PRPn [c0] --> NADP [c0] + L-Methionine [c0] + Methane [c0] + 5'-Deoxyadenosine [c0] + 5-phospho-alpha-D-ribosyl 1,2-cyclic phosphate [c0]"
GPR,Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.16.contigs__.RAST.CDS.2379_mRNA or...
Lower bound,0
Upper bound,1000


In [59]:
model.metabolites.cpd26508_c0

Metabolite identifier,cpd26508_c0
Name,"5-phospho-alpha-D-ribosyl 1,2-cyclic phosphate..."
Memory address,0x7f2ba9c64910
Formula,C5H7O10P2
Compartment,c0
In 1 reaction(s),rxn43120_c0


In [43]:
template = kbase.get_from_ws(model.template_ref)

In [48]:
model.reactions.SK_cpd03091_c0 

Reaction identifier,SK_cpd03091_c0
Name,Sink for 5'-Deoxyadenosine [c0]
Memory address,0x7f2ba88c0460
Stoichiometry,cpd03091_c0 --> 5'-Deoxyadenosine [c0] -->
GPR,
Lower bound,0
Upper bound,1000


In [46]:
template.drains

{<NewModelTemplateCompCompound cpd01042_c at 0x7f2ba59be710>: (0, 1000),
 <NewModelTemplateCompCompound cpd02701_c at 0x7f2ba58a87f0>: (0, 1000),
 <NewModelTemplateCompCompound cpd03091_c at 0x7f2ba5b403a0>: (0, 1000),
 <NewModelTemplateCompCompound cpd11416_c at 0x7f2ba5b41720>: (0, 1000),
 <NewModelTemplateCompCompound cpd15302_c at 0x7f2ba5a73a00>: (0, 1000),
 <NewModelTemplateCompCompound cpd22811_c at 0x7f2ba56c53c0>: (0, 1000)}

In [50]:
for r in model.metabolites.cpd03091_c0.reactions:
    print(r.id, '\t', r.build_reaction_string(True))

rxn04704_c0 	 2 S-Adenosyl-L-methionine [c0] + CoproporphyrinogenIII [c0] --> 2 CO2 [c0] + 2 L-Methionine [c0] + ProtoporphyrinogenIX [c0] + 2 5'-Deoxyadenosine [c0]
rxn40559_c0 	 S-Adenosyl-L-methionine [c0] + 13(1)-Hydroxy-Mg-protoporphyrin IX 13-monomethyl ester [c0] --> L-Methionine [c0] + 3 H+ [c0] + 5'-Deoxyadenosine [c0] + 13(1)-Oxo-Mg-protoporphyrin IX 13-monomethyl ester [c0]
rxn46959_c0 	 H2O [c0] + S-Adenosyl-L-methionine [c0] + Mg-Protoporphyrin IX 13-monomethyl ester [c0] <=> L-Methionine [c0] + H+ [c0] + 5'-Deoxyadenosine [c0] + 13(1)-Hydroxy-Mg-protoporphyrin IX 13-monomethyl ester [c0]
SK_cpd03091_c0 	 5'-Deoxyadenosine [c0] --> 
rxn43120_c0 	 NADPH [c0] + S-Adenosyl-L-methionine [c0] + PRPn [c0] --> NADP [c0] + L-Methionine [c0] + Methane [c0] + 5'-Deoxyadenosine [c0] + 5-phospho-alpha-D-ribosyl 1,2-cyclic phosphate [c0]
rxn43329_c0 	 S-Adenosyl-L-methionine [c0] + 13(1)-Oxo-Mg-protoporphyrin IX 13-monomethyl ester [c0] <-- L-Methionine [c0] + 5'-Deoxyadenosine [c0] + 

In [56]:
model.objective = 'rxn43120_c0'

In [65]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00001_e0,EX_cpd00001_e0,166.7,0,0.00%
cpd00007_e0,EX_cpd00007_e0,166.7,0,0.00%
cpd11640_e0,EX_cpd11640_e0,333.3,0,0.00%
cpd25960_c0,EX_cpd25960_c0,1000,1,100.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00012_e0,EX_cpd00012_e0,-500,0,0.00%
cpd00067_e0,EX_cpd00067_e0,-500,0,0.00%
cpd01024_e0,EX_cpd01024_e0,-1000,1,100.00%


In [7]:
model.metabolites.cpd26519_c0

Metabolite identifier,cpd26519_c0
Name,PRPn [c0]
Memory address,0x7f2ba9c64490
Formula,C6H11O10P2
Compartment,c0
In 2 reaction(s),"rxn26457_c0, rxn43120_c0"


In [51]:
model.metabolites.cpd00128_c0

Metabolite identifier,cpd00128_c0
Name,Adenine [c0]
Memory address,0x7f2baa396560
Formula,C5H5N5
Compartment,c0
In 6 reaction(s),"rxn00131_c0, rxn26456_c0, rxn00139_c0, rxn01022_c0, rxn00926_c0, rxn00927_c0"


In [53]:
if 'cpd00128_c0' in model.metabolites and 'SK_cpd00128_c0' not in model.reactions:
    ex = Reaction('SK_cpd00128_c0', 'Sk for adenine', '', 0, 1000)
    ex.add_metabolites({
        model.metabolites.cpd00128_c0: -1
    })
    model.add_reactions([ex])
    print('!')

In [11]:
model.reactions.rxn26456_c0

Reaction identifier,rxn26456_c0
Name,RXN0-6732.c [c0]
Memory address,0x7f2ba9010100
Stoichiometry,cpd00002_c0 + cpd25960_c0 --> cpd00128_c0 + cpd26518_c0 ATP [c0] + MePn [c0] --> Adenine [c0] + RPnTP [c0]
GPR,(Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.16.contigs__.RAST.CDS.2377 or...
Lower bound,0
Upper bound,1000


Metabolite identifier,cpd25960_c0
Name,MePn [c0]
Memory address,0x7f2ba9c17940
Formula,CH4O3P
Compartment,c0
In 1 reaction(s),rxn26456_c0


In [8]:
model.reactions.rxn26457_c0

Reaction identifier,rxn26457_c0
Name,RXN0-6733.c [c0]
Memory address,0x7f2ba8ed4640
Stoichiometry,cpd00001_c0 + cpd26518_c0 --> cpd00012_c0 + 2 cpd00067_c0 + cpd26519_c0 H2O [c0] + RPnTP [c0] --> PPi [c0] + 2 H+ [c0] + PRPn [c0]
GPR,Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.16.contigs__.RAST.CDS.2384 or...
Lower bound,0
Upper bound,1000


In [5]:
model.reactions.rxn43120_c0

Reaction identifier,rxn43120_c0
Name,rxn43120_c [c0]
Memory address,0x7f2ba8f1d360
Stoichiometry,"cpd00005_c0 + cpd00017_c0 + cpd26519_c0 --> cpd00006_c0 + cpd00060_c0 + cpd01024_c0 + cpd03091_c0 + cpd26508_c0 NADPH [c0] + S-Adenosyl-L-methionine [c0] + PRPn [c0] --> NADP [c0] + L-Methionine [c0] + Methane [c0] + 5'-Deoxyadenosine [c0] + 5-phospho-alpha-D-ribosyl 1,2-cyclic phosphate [c0]"
GPR,Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.16.contigs__.RAST.CDS.2379_mRNA or...
Lower bound,0
Upper bound,1000


In [2]:
ws_reg = 155805
ws_enh = 186678

In [3]:
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')

In [98]:
model_reg = kbase.get_from_ws('Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.37.contigs__.RAST.pyr.O2', ws_reg)
model_enh = kbase.get_from_ws('Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.37.pangenome.RAST.glm4ec.pyr.O2', ws_enh)

Set parameter TSPort to value 27070
Set parameter TokenServer to value "lic-vmw-01.cels.anl.gov"


In [66]:
ws_enh2 = 188065
models_enh2 = {}
for info in kbase.list_workspace(ws_enh2):
    if info.type == 'KBaseFBA.FBAModel' and info.id.endswith('.pyr.O2'):
        models_enh2[info.id] = kbase.get_object(info.id, ws_enh2)
        print(info.id, info.type)

R1_A_D2.21.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R1_A_D1.5.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R1_A_H2O.14.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R1_A_H2O.8.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R1_B_H2O.17.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R1_C_D1.31.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R1_C_H2O.29.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_D2.19.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_D2.9.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_D2.67.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_D2.14.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_D2.16.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_H2O.11.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_H2O.29.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_H2O.45.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_H2O.14.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_A_H2O.035.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_B_H2O.16.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_B_H2O.17.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_C_H2O.35.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_C_H2O.20.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_restored_DShore.12.pg.G.D.pyr.O2 KBaseFBA.FBAModel
R2_restored_DShore.37.pg.G.D.pyr

In [4]:
models_reg = {}
for info in kbase.list_workspace(ws_reg):
    if info.type == 'KBaseFBA.FBAModel' and info.id.endswith('.pyr.O2'):
        models_reg[info.id] = kbase.get_object(info.id, ws_reg)
        print(info.id, info.type)

Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.11.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.5.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.39.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.119.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.21.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.39.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_C_D2_MG_DASTool_bins_concoct_out.47.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2A_A_H2O_MG_DASTool_bins_concoct_out.78.contigs__.RAST.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaGSF2_A_H2O_MG_DASTool_bins_metab

In [5]:
models_enh = {}
for info in kbase.list_workspace(ws_enh):
    if info.type == 'KBaseFBA.FBAModel' and info.id.endswith('.pyr.O2'):
        models_enh[info.id] = kbase.get_object(info.id, ws_enh)
        print(info.id, info.type)

Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.17.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.9.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.14.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_metabat.31.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.8.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.5.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.67.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.19.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_C_H2O_MG_DASTool_bins_metabat.29.pangenome.RAST.glm4ec.pyr.O2 KBaseFBA.FBAModel

In [ ]:
for info in kbase.list_workspace(ws_enh):
    if info.type == 'KBaseFBA.FBAModel':
        kbase.get_from_ws(info.id, ws_enh)
        print(info.id, info.type)

In [78]:
import pandas as pd
mapping = pd.read_csv('./mag_name_mapping.tsv', header=None, sep='\t', index_col=0).to_dict()[1]

In [35]:
templates = {}

In [38]:
for k in templates:
    print(templates[k].info.id)

template_v6gn_methylphosphonate_deg
template_v6gn_methylphosphonate_deg
ArchaeaTemplateV5
template_v6gp_methylphosphonate_deg
template_v6gp_methylphosphonate_deg


In [96]:
bad_mapping

{'Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.8.contigs__.RAST.pyr.O2': AttributeError("'float' object has no attribute 'replace'")}

In [95]:
bad_mapping = {}
for k in models_reg:
    try:
        model_reg = models_reg[k]
        model_enh = models_enh2.get(mapping[k].replace('.pg.', '.pg.G.D.'))
        if model_enh is not None:
            model_reg_template = model_reg['template_ref']
            if model_reg_template not in templates:
                templates[model_reg_template] = kbase.get_from_ws(model_reg_template)
            model_enh_template = model_enh['template_ref']
            if model_enh_template not in templates:
                templates[model_enh_template] = kbase.get_from_ws(model_enh_template)
            reg_gapfill = _get_gapfilled_reactions(model_reg, templates[model_reg['template_ref']])
            enh_gapfill = _get_gapfilled_reactions(model_enh, templates[model_enh['template_ref']])
            d = [k, len(model_reg['modelreactions']), len(model_enh['modelreactions']), len(reg_gapfill), len(enh_gapfill), len(_get_all_genes(model_reg)), len(_get_all_genes(model_enh))]
            print('\t'.join([str(x) for x in d]))
    except Exception as ex:
        bad_mapping[k] = ex

Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.11.contigs__.RAST.pyr.O2	957	957	71	38	1165	885
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.5.contigs__.RAST.pyr.O2	831	831	110	72	1141	980
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.contigs__.RAST.pyr.O2	679	679	86	62	957	687
Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST.pyr.O2	677	677	126	104	814	507
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.39.contigs__.RAST.pyr.O2	1174	1174	60	45	1585	1039
Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.119.contigs__.RAST.pyr.O2	918	918	167	134	1046	755
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.21.contigs__.RAST.pyr.O2	981	981	58	40	1549	1193
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.39.contigs__.RAST.pyr.O2	871	871	159	126	847	541
Salt_Pond_MetaG_R2_C_D2_MG_DASTool_bins_concoct_out.47.contigs__.RAST.pyr.O2	1210	1210	69	45	2361	1794
Salt_Pond_MetaG_R2A_A_H2O_MG_DASTool_bins_concoct_out.78.contigs__.RAST.pyr.O2	878	878	

In [62]:
for k in models_reg:
    model_reg = models_reg[k]
    model_enh = models_enh[k.split('contigs__')[0] + 'pangenome.RAST.glm4ec.pyr.O2']
    model_reg_template = model_reg['template_ref']
    if model_reg_template not in templates:
        templates[model_reg_template] = kbase.get_from_ws(model_reg_template)
    model_enh_template = model_enh['template_ref']
    if model_enh_template not in templates:
        templates[model_enh_template] = kbase.get_from_ws(model_enh_template)
    reg_gapfill = _get_gapfilled_reactions(model_reg, templates[model_reg['template_ref']])
    enh_gapfill = _get_gapfilled_reactions(model_enh, templates[model_enh['template_ref']])
    d = [k, len(model_reg['modelreactions']), len(model_enh['modelreactions']), len(reg_gapfill), len(enh_gapfill), len(_get_all_genes(model_reg)), len(_get_all_genes(model_enh))]
    print('\t'.join([str(x) for x in d]))

Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.11.contigs__.RAST.pyr.O2	957	972	71	54	1165	1247
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.concoct_out.5.contigs__.RAST.pyr.O2	831	840	110	93	1141	1252
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.contigs__.RAST.pyr.O2	679	686	86	68	957	1046
Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST.pyr.O2	677	684	126	108	814	909
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.39.contigs__.RAST.pyr.O2	1174	1187	60	54	1585	1697
Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.119.contigs__.RAST.pyr.O2	918	935	167	141	1046	1194
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.21.contigs__.RAST.pyr.O2	981	996	58	49	1549	1640
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.39.contigs__.RAST.pyr.O2	871	877	159	150	847	887
Salt_Pond_MetaG_R2_C_D2_MG_DASTool_bins_concoct_out.47.contigs__.RAST.pyr.O2	1210	1218	69	54	2361	2603
Salt_Pond_MetaG_R2A_A_H2O_MG_DASTool_bins_concoct_out.78.contigs__.RAST.pyr.O2	878	

957

In [20]:
model_enh['modelreactions'][0]['modelReactionProteins']

[{'complex_ref': '~/template/complexes/name/cpx00700',
  'modelReactionProteinSubunits': [{'feature_refs': ['~/genome/features/id/Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.38.contigs__.RAST.CDS.1589',
     '~/genome/features/id/Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.38.contigs__.RAST.CDS.1589_mRNA'],
    'note': 'Dihydropteroate synthase (EC 2.5.1.15)',
    'optionalSubunit': 0,
    'role': 'ftr01608',
    'triggering': 1}],
  'note': '',
  'source': 'ModelSEED'},
 {'complex_ref': '~/template/complexes/name/cpx01370',
  'modelReactionProteinSubunits': [{'feature_refs': ['~/genome/features/id/Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.38.contigs__.RAST.CDS.1507',
     '~/genome/features/id/Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.38.contigs__.RAST.CDS.1507_mRNA'],
    'note': '2-amino-4-hydroxy-6-hydroxymethyldihydropteridine pyrophosphokinase (EC 2.7.6.3)',
    'optionalSubunit': 0,
    'role': 'ftr01607',
    'triggering': 1}],
  'note': '',
  'source': 'Mod

In [29]:
def _get_features(r):
    res = set()
    for cpx in r['modelReactionProteins']:
        for su in cpx['modelReactionProteinSubunits']:
            for f in su['feature_refs']:
                res.add(f.split('/')[-1])
    return res
def _get_all_genes(m):
    all_genes = set()
    for r in m['modelreactions']:
        all_genes |= _get_features(r)
    return all_genes

In [60]:
def _get_gapfilled_reactions(m, t):
    gapfilled = set()
    for r in m['modelreactions']:
        cpx_count = len(r['modelReactionProteins'])
        if cpx_count == 0:
            template_rxn = t.reactions.get_by_id(r['reaction_ref'].split('/')[-1])
            if template_rxn.type == 'conditional' or template_rxn.type == 'gapfilling':
                gapfilled.add(r['id'])
    return gapfilled
len(_get_gapfilled_reactions(model_reg, templates[model_reg['template_ref']]))

64